# Week 5-1 · PBQ-01 — Basics of Python Programming
**The first official coding lecture — Python from zero, tailored for quant trading.**

This is a hands-on primer. Every concept is shown by a runnable example, exactly as in the lecture's
Jupyter notebook, and we finish where the lecture finishes: **turning Python into trading signals** on
real **TCS stock data** (`TCS.NS.csv`, 2,460 daily bars).

**What we cover (each with a worked example):**
1. `print`, **f-strings**, and `.format` (the Intel market-cap example)
2. Variables & **dynamic typing**
3. The three native **data structures**: lists, tuples, dictionaries
4. **Indexing & slicing**
5. **NumPy**: arrays, vectorised maths, `np.random.normal`, `np.where`
6. **pandas**: `read_csv`, `shape`, `head`, `describe` on real TCS data
7. **Loops** (`range`, `enumerate`) & **conditionals**
8. **Trading signals** — vectorised and with a for-loop, then a moving-average crossover

> Why learn Python when AI writes code? Because you must *verify* that a generated strategy's logic is
> actually correct — the constraints, the entry/exit, the edge cases. That requires reading code fluently.


In [1]:
# A standard Python program starts with imports
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
print("numpy", np.__version__, "| pandas", pd.__version__)


numpy 2.3.5 | pandas 3.0.3


## Part 1 — `print`, f-strings, and `.format`

`print` gives you control over how output *reads*. A bare number tells you nothing; labelled output tells
a story. Three styles, same result — the lecture's Intel market-cap example.

In [2]:
ticker     = "INTC"
firm_name  = "Intel Corporation"
last_price = 25.47
date_price = "22-Feb-2023"
market_cap = 105.95 * 10**9          # 105.95 billion
shares_out = market_cap / last_price

# style 1: comma-separated strings + values
print("The market capitalization of", firm_name, "at the close of", date_price,
      "is US$", market_cap)

# style 2: f-string (cleanest)
print(f"We display the ticker as {ticker} and the share price as US$ {last_price}")

# style 3: .format with index + type + precision
print("{0:s} has {1:.1f} million outstanding shares".format(firm_name, shares_out/10**6))


The market capitalization of Intel Corporation at the close of 22-Feb-2023 is US$ 105950000000.0
We display the ticker as INTC and the share price as US$ 25.47
Intel Corporation has 4159.8 million outstanding shares


### How `.format` maps its arguments
The placeholder `{0:s}` means *take argument 0, render as string*; `{1:.1f}` means *take argument 1,
render as float with 1 decimal*. Indexing starts at **0** in Python.

In [3]:
print("STEP TABLE — {index:type} -> which argument, how rendered")
print(f"  {{0:s}}   -> arg 0 '{firm_name}'  as string")
print(f"  {{1:.1f}} -> arg 1  {shares_out/10**6:.4f}  rounded to 1 dp -> {shares_out/10**6:.1f}")


STEP TABLE — {index:type} -> which argument, how rendered
  {0:s}   -> arg 0 'Intel Corporation'  as string
  {1:.1f} -> arg 1  4159.7958  rounded to 1 dp -> 4159.8


## Part 2 — Variables & dynamic typing

Python decides a variable's type from the value you assign — no declaration needed. `=` is *assignment*,
not equality.

In [4]:
x = 5;     print("x =", x, "-> type", type(x).__name__)
x = 5.5;   print("x =", x, "-> type", type(x).__name__)
x = "five";print("x =", x, "-> type", type(x).__name__)


x = 5 -> type int
x = 5.5 -> type float
x = five -> type str


## Part 3 — The three native data structures

| structure | brackets | mutable? | use |
|---|---|---|---|
| **list** | `[ ]` | yes | ordered, changeable collection |
| **tuple** | `( )` | no (immutable) | fixed collection |
| **dict** | `{ }` | yes | key → value lookup |

All can hold *heterogeneous* elements (strings, floats, ints mixed).

In [5]:
# LIST — ordered & changeable
exchanges = ["NSE","NYSE","NASDAQ","LSE","SGX","HKEX"]
exchanges.append("EURONEXT")          # add at end
exchanges.insert(0,"TYO")             # add at position 0
print("list after append+insert:", exchanges, "| len =", len(exchanges))

# TUPLE — immutable
t_exch = ("NSE","NYSE","NASDAQ")
print("tuple:", t_exch, "| append exists?", hasattr(t_exch,"append"))

# DICT — key:value
prices = {"TSLA":248.5, "INTC":25.47, "AAPL":189.9}
print("dict keys:", list(prices.keys()))
print("dict values:", list(prices.values()))
print("price of INTC via key:", prices["INTC"])


list after append+insert: ['TYO', 'NSE', 'NYSE', 'NASDAQ', 'LSE', 'SGX', 'HKEX', 'EURONEXT'] | len = 8
tuple: ('NSE', 'NYSE', 'NASDAQ') | append exists? False
dict keys: ['TSLA', 'INTC', 'AAPL']
dict values: [248.5, 25.47, 189.9]
price of INTC via key: 25.47


## Part 4 — Indexing & slicing

Left-to-right indices start at `0`; right-to-left start at `-1`. A slice `a:b` includes `a` and
**excludes** `b`.

In [6]:
bp = ["Oil, Gas and Coal", "Energy", 290.0, "LSE", "GBp"]
print("bp[0]   =", bp[0])        # first
print("bp[-1]  =", bp[-1])       # last
print("bp[1:3] =", bp[1:3])      # index 1,2 (3 excluded)
print("'LSE' in bp ->", "LSE" in bp)   # membership test


bp[0]   = Oil, Gas and Coal
bp[-1]  = GBp
bp[1:3] = ['Energy', 290.0]
'LSE' in bp -> True


## Part 5 — NumPy: arrays & vectorisation

A NumPy **array** holds *homogeneous* elements and supports element-wise maths — the key reason it beats
a plain list for quant work. Watch the difference:

In [7]:
heights = [1.65, 1.80, 1.72, 1.90, 1.55]      # a list
print("list  * 2 ->", heights*2)              # DUPLICATES the list!
arr = np.array(heights)
print("array * 2 ->", arr*2)                  # multiplies each element
print("max =", arr.max(), "| mean =", np.mean(arr), "| median =", np.median(arr))


list  * 2 -> [1.65, 1.8, 1.72, 1.9, 1.55, 1.65, 1.8, 1.72, 1.9, 1.55]
array * 2 -> [3.3  3.6  3.44 3.8  3.1 ]
max = 1.9 | mean = 1.7240000000000002 | median = 1.72


### Random numbers & `np.where`
`np.random.normal(loc, scale, size)` draws from a normal distribution (`loc`=mean, `scale`=std). With one
line you generate thousands of points — impossible to do comfortably in a spreadsheet. `np.where` finds
the indices where a condition holds.

In [8]:
rng = np.random.default_rng(7)
sample = rng.normal(loc=0, scale=1, size=10000)
print("generated", len(sample), "points | mean ~", round(sample.mean(),3),
      "| std ~", round(sample.std(),3))

vol = np.round(rng.normal(250, 50, 100), 2)        # 100 fake volumes
low_idx = np.where(vol < 150)[0]
print("\nindices where volume < 150:", low_idx)
print("the values there:", vol[low_idx])


generated 10000 points | mean ~ -0.012 | std ~ 0.994

indices where volume < 150: [47]
the values there: [138.69]


## Part 6 — pandas on real TCS data

`pandas` is the workhorse for loading and shaping market data. We read the TCS daily file, make the date
the index, and inspect it.

In [9]:
df = pd.read_csv("TCS.NS.csv", index_col="Date", parse_dates=["Date"], dayfirst=True)
print("shape (rows, cols):", df.shape)
print("\nfirst 3 rows:")
print(df.head(3).to_string())
print("\ndescribe (Close):")
print(df["Close"].describe().round(2).to_string())


shape (rows, cols): (2460, 6)

first 3 rows:
                  High         Low        Open       Close   Volume   Adj Close
Date                                                                           
2010-04-26  395.500000  390.100006  393.000000  392.549988  2512548  294.111939
2010-04-27  394.950012  390.524994  392.975006  393.649994  2621234  294.936157
2010-04-28  391.975006  382.500000  390.000000  385.149994  2400382  288.567657

describe (Close):
count    2460.00
mean     1170.19
std       523.20
min       349.77
25%       645.58
50%      1198.76
75%      1334.99
max      2277.95


## Part 7 — Loops & conditionals

A `for` loop repeats an action over a sequence. `range(n)` yields `0..n-1`. `enumerate` gives you both the
**index** and the **value** — useful when you need *which day* a price condition fired.

In [10]:
for i in range(5):
    print(f"  i={i}  ->  i*2 = {i*2}")

print("\nenumerate (index + value):")
for day, price in enumerate([70, 84, 66, 88, 75]):
    flag = "SELL" if price > 83 else ("BUY" if price < 67 else "-")
    print(f"  day {day}: price {price} -> {flag}")


  i=0  ->  i*2 = 0
  i=1  ->  i*2 = 2
  i=2  ->  i*2 = 4
  i=3  ->  i*2 = 6
  i=4  ->  i*2 = 8

enumerate (index + value):
  day 0: price 70 -> -
  day 1: price 84 -> SELL
  day 2: price 66 -> BUY
  day 3: price 88 -> SELL
  day 4: price 75 -> -


## Part 8 — Turning Python into trading signals

The lecture's payoff. First the **vectorised** way (`np.where`, nested for buy/sell), then a real
**moving-average crossover** on TCS — the whole point of the course.

In [11]:
# vectorised signals on volume: short if high volume, long if low, else flat
vol2 = np.round(rng.normal(250, 50, 100), 2)
signals = np.where(vol2 > 325, "short", "0")
signals = np.where(vol2 < 175, "long", signals)
vals, counts = np.unique(signals, return_counts=True)
print("signal counts:", dict(zip(vals, counts)))


signal counts: {np.str_('0'): np.int64(87), np.str_('long'): np.int64(7), np.str_('short'): np.int64(6)}


In [12]:
# real MA crossover on TCS: long when SMA20 > SMA50
d = df.sort_index().copy()
d["SMA20"] = d["Close"].rolling(20).mean()
d["SMA50"] = d["Close"].rolling(50).mean()
d["signal"] = np.where(d["SMA20"] > d["SMA50"], 1, 0)   # 1 = long, 0 = flat
d["position"] = d["signal"].shift(1).fillna(0)
crossings = int((d["signal"].diff()==1).sum())
print(f"TCS bars: {len(d)} | long-entry crossovers: {crossings}")
print(f"days in market: {int(d['position'].sum())} / {len(d)} "
      f"({d['position'].mean():.0%})")


TCS bars: 2460 | long-entry crossovers: 34
days in market: 1456 / 2460 (59%)


In [13]:
fig, ax = plt.subplots(figsize=(9,3.6))
ax.plot(d.index, d["Close"], color="#cbd5e1", lw=.7, label="TCS close")
ax.plot(d.index, d["SMA20"], color="#0ea5e9", lw=1, label="SMA 20")
ax.plot(d.index, d["SMA50"], color="#ef4444", lw=1, label="SMA 50")
ax.fill_between(d.index, d["Close"].min(), d["Close"].max(), where=d["position"]==1,
                color="#86efac", alpha=.18, label="long")
ax.set_title("TCS — moving-average crossover signals"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig("chart_1_tcs_signals.png", dpi=110); plt.show()
print("saved chart_1_tcs_signals.png")


saved chart_1_tcs_signals.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_30636\761334292.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_1_tcs_signals.png", dpi=110); plt.show()


## Summary — your Python starter kit
- **Output:** `print`, f-strings (`f"{var}"`), and `.format` with `{index:type.precision}`.
- **Typing is dynamic:** `=` assigns; the value decides the type.
- **Data structures:** list `[]` (mutable), tuple `()` (immutable), dict `{}` (key→value). Indexing from **0**; slices exclude the end.
- **NumPy** arrays vectorise maths (a list `*2` duplicates; an array `*2` scales). `np.where` locates conditions; `np.random.normal` simulates.
- **pandas** `read_csv` loads market data; `shape`/`head`/`describe` inspect it.
- **Loops:** `for ... range(n)`, `enumerate` for index+value; `if/elif/else` for branching.
- **Signals** are just conditions turned into +1/0/−1 — the foundation of every strategy in this course.

> Practice ~1 hour a day: pick one topic, type it, run it. Coding is muscle memory.
